[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RobMulla/torchtpu-inference-workshop/blob/main/notebooks/01_torchtpu_one_chip.ipynb)

# Demo 1: TorchTPU on one chip

The first live demo of the workshop, on one chip: set the device, run a model in eager mode, then with DeferAndFuse, then under `torch.compile`, and look at one step of each under the profiler.

**Runtime:** in Colab, *Runtime → Change runtime type → TPU*. About five minutes.

![TorchTPU makes the TPU a PyTorch device: eager, DeferAndFuse, torch.compile](https://raw.githubusercontent.com/RobMulla/torchtpu-inference-workshop/main/notebooks/img/slide_03.jpg)

## 1. Install

`torch_tpu` from PyPI. One package; nothing else to set up.

In [ ]:
import importlib.util, os

if importlib.util.find_spec("google.colab"):  # Colab's TPU runtime: one chip, no metadata server
    for k, v in {"TPU_SKIP_MDS_QUERY": "1", "TPU_CHIPS_PER_HOST_BOUNDS": "1,1,1",
                 "TPU_HOST_BOUNDS": "1,1,1", "TPU_WORKER_ID": "0", "TPU_WORKER_HOSTNAMES": "localhost"}.items():
        os.environ.setdefault(k, v)  # inherited by the scripts and servers started below

import importlib.util, importlib.metadata as md

if importlib.util.find_spec("torch_tpu") is None:
    %pip install -q torch_tpu==0.2.0 --extra-index-url https://download.pytorch.org/whl/cpu

for p in ["torch", "torch_tpu", "libtpu"]:
    print(f"{p:10s} {md.version(p)}")

## 2. The device line

On a TPU VM, `import torch` is enough: TorchTPU registers the `tpu` device when torch loads. Colab's sandbox hides the PCI bus that TorchTPU scans for the chip, so the cell registers the device explicitly there (two lines that do nothing on a VM). That one string, `"tpu"`, is the whole change from a CPU script.

In [ ]:
import torch  # TorchTPU loads with torch; on a TPU VM that alone registers the "tpu" device
from torch_tpu import _loader
if not _loader._LOADED:  # a sandbox that hides /sys/bus/pci (Colab's): the scan found no TPU, register it anyway
    _loader._init_device("tpu")

device = torch.accelerator.current_accelerator()
print("current accelerator:", device)

x = torch.randn(4, 4, device="tpu")
print(x.device, x.dtype, tuple(x.shape))

## 3. Eager mode

A 12-block MLP and a forward pass in a loop, which is what an engine runs thousands of times. Eager runs one op at a time, in the order the Python runs: the mode to debug in. Read the step time off the output.

In [ ]:
import time
from torch import nn

model = nn.Sequential(*[
    nn.Sequential(nn.LayerNorm(1024),
                  nn.Linear(1024, 4096), nn.GELU(),
                  nn.Linear(4096, 1024))
    for _ in range(12)
]).to(device)
x = torch.randn(16, 512, 1024, device=device)

with torch.no_grad():
    for i in range(1, 301):
        t0 = time.perf_counter()
        y = model(x)
        torch.accelerator.synchronize()  # wait
        ms = (time.perf_counter() - t0) * 1000
        if i == 1 or i % 100 == 0:
            print(f"eager step {i}: {ms:.1f} ms")

print(y.device, tuple(y.shape))

## 4. DeferAndFuse

Same model, same loop, one setting: TorchTPU holds the ops back and runs them in fused blocks. Still eager (you can still print a tensor), and marked experimental. The live demo did the same with `TORCH_TPU_DEFER_AND_FUSE=1` on the command line.

In [ ]:
def steps(fn, label, n=300):
    with torch.no_grad():
        for i in range(1, n + 1):
            t0 = time.perf_counter()
            y = fn(x)
            torch.accelerator.synchronize()  # wait
            ms = (time.perf_counter() - t0) * 1000
            if i == 1 or i % 100 == 0:
                print(f"{label} step {i}: {ms:.1f} ms")
    return y


with torch.tpu.set_eager_mode(torch.tpu.EagerMode.DEFER_AND_FUSE):
    y = steps(model, "defer")
print(y.device, tuple(y.shape))

## 5. `torch.compile`

One line: the whole step becomes one program for the TPU. The first step pays the compile; then compare the step time with the two above.

In [ ]:
compiled = torch.compile(model, backend="tpu")

y = steps(compiled, "compiled")
print(y.device, tuple(y.shape))

## 6. The three runs under the profiler

The standard `torch.profiler` with the TPU activity turned on; five steps of each, written to `traces/`. They open in TensorBoard with the XProf plugin; the next cell draws the one picture that matters.

In [ ]:
import os
from torch.profiler import ProfilerActivity, profile, tensorboard_trace_handler
from torch.tpu.profiler import TpuProfilerConfig


def capture(name, fn, n=5):
    """n steps of fn(x) under the profiler, written to traces/<name>."""
    trace_dir = os.path.abspath(f"traces/{name}")
    with torch.no_grad(), profile(
        activities=[ProfilerActivity.CPU, ProfilerActivity.TPU],
        experimental_config=TpuProfilerConfig(run_dir=trace_dir),
        schedule=torch.profiler.schedule(wait=0, warmup=0, active=n, repeat=1),
        on_trace_ready=tensorboard_trace_handler(trace_dir),
    ) as prof:
        for i in range(1, n + 1):
            t0 = time.perf_counter()
            y = fn(x)
            torch.accelerator.synchronize()  # wait
            print(f"{name} step {i}: {(time.perf_counter() - t0) * 1000:.1f} ms")
            prof.step()  # marks the step in the trace (ProfilerStep#n)
    for root, _, files in os.walk(trace_dir):
        for f in sorted(files):
            p = os.path.join(root, f)
            print(f"  {os.path.relpath(p, 'traces')} ({os.path.getsize(p) / 1024:.0f} KB)")
    return trace_dir


eager_dir = capture("eager", model)
with torch.tpu.set_eager_mode(torch.tpu.EagerMode.DEFER_AND_FUSE):
    defer_dir = capture("defer", model)
compiled_dir = capture("compiled", compiled)

## 7. One step of each, same scale

Every bar is one program the TPU ran. Eager: a comb of small programs, one per op. DeferAndFuse: fewer, bigger blocks. Compiled: one. Same math in all three; the mode changes the shape, and the shape is the time.

In [ ]:
import glob
import json

if importlib.util.find_spec("matplotlib") is None:
    %pip install -q matplotlib
import matplotlib.pyplot as plt


def tpu_programs(trace_dir, step_index=2):
    """The TPU programs of one step: [(start_ms, dur_ms, name)], and the step's length in ms."""
    path = max(glob.glob(f"{trace_dir}/*.pt.trace.json"), key=os.path.getmtime)
    with open(path) as f:
        events = json.load(f)["traceEvents"]
    threads = {(e["pid"], e["tid"]): e["args"]["name"]
               for e in events if e.get("ph") == "M" and e["name"] == "thread_name"}
    programs = [e for e in events if e.get("ph") == "X"
                and threads.get((e["pid"], e["tid"])) == "XLA Modules"]
    steps = sorted((e for e in events if e.get("ph") == "X"
                    and e["name"].startswith("ProfilerStep#")), key=lambda e: e["ts"])
    if steps:  # one steady step; the profiler marks each prof.step()
        s = steps[min(step_index, len(steps) - 1)]
        t0, t1 = s["ts"], s["ts"] + s["dur"]
    else:
        t0 = min(e["ts"] for e in programs)
        t1 = max(e["ts"] + e["dur"] for e in programs)
    inside = sorted((e for e in programs if t0 <= e["ts"] <= t1), key=lambda e: e["ts"])
    return [((e["ts"] - t0) / 1000, e["dur"] / 1000, e["name"]) for e in inside], (t1 - t0) / 1000


rows = [("eager", "#4285F4", *tpu_programs(eager_dir)),
        ("DeferAndFuse", "#F9AB00", *tpu_programs(defer_dir)),
        ("compiled", "#34A853", *tpu_programs(compiled_dir))]
width = max(step_ms for _, _, _, step_ms in rows)

for label, _, bars, step_ms in rows:
    print(f"{label:13s} {len(bars)} program{'s' if len(bars) != 1 else ''} on the TPU in one step of {step_ms:.1f} ms")

fig, ax = plt.subplots(figsize=(14, 3.6))
for y, (label, color, bars, step_ms) in enumerate(reversed(rows)):
    ax.broken_barh([(s, max(d, width / 1400)) for s, d, _ in bars], (y + 0.15, 0.7),
                   facecolor=color, edgecolor="white", linewidth=0.8)
    n = f"{len(bars)} program{'s' if len(bars) != 1 else ''}"
    ax.text(width * 1.01, y + 0.5, f"{n}, {step_ms:.1f} ms", va="center")
ax.set_xlim(0, width)
ax.set_yticks([0.5, 1.5, 2.5], [r[0] for r in reversed(rows)])
ax.set_xlabel("ms from the start of the step (same scale for all three)")
ax.set_title("What the TPU ran in one step: one op at a time, fused blocks, or one program")
plt.tight_layout()
plt.show()

The session's version of the same picture, from the trace viewer:

![Profile results: one step of eager, DeferAndFuse and compiled, same scale](https://raw.githubusercontent.com/RobMulla/torchtpu-inference-workshop/main/notebooks/img/slide_05.jpg)

Debug in eager, speed up with one setting, ship compiled.

### Opening the traces yourself

On a TPU VM: `pip install tensorboard xprof`, then `tensorboard --logdir traces`, Profile tab, Trace Viewer. The *XLA Modules* row is the programs the chip ran; *XLA Ops* the ops inside them.

## Next

- TorchTPU: [github.com/google-pytorch/torch_tpu](https://github.com/google-pytorch/torch_tpu) (README; `docs/performance_profiling_with_xprof.md`).
- The live demo's files: [`demos/01_one_chip/`](https://github.com/RobMulla/torchtpu-inference-workshop/tree/main/demos/01_one_chip).
- Demos 2 and 3 as notebooks: `02_sglang_torchtpu.ipynb`, `03_vllm_torchtpu.ipynb`, or all three in `00_all_three_demos.ipynb`.